# Week 14: Training AI Models — Fine-Tuning a Fraud Classifier

## Learning Objectives

By the end of this session, you will be able to:
1. **Prepare text data** for fine-tuning with HuggingFace tokenizers and datasets
2. **Fine-tune DistilBERT** for binary classification using transfer learning
3. **Evaluate fine-tuned models** against prompted models from previous weeks
4. **Decide when to fine-tune** vs when prompt engineering is sufficient

## Prerequisites

- Completed Weeks 11-13 (LLM APIs, local models, evaluation, Bedrock, synthetic data)
- Watched pre-class videos on fine-tuning concepts, transfer learning, LoRA/QLoRA theory
- Synthetic training data from Week 13 Lab 3 (or we'll generate fallback data)

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup & Load Data | 10 min | Code |
| Section 1: Transfer Learning Concepts | 15 min | Theory + Demo |
| Section 2: Tokenization & Data Prep | 20 min | Demo |
| Lab 1: Prepare Your Dataset | 15 min | Lab |
| Section 3: Fine-Tuning DistilBERT | 25 min | Demo-heavy |
| Lab 2: Fine-Tune Your Classifier | 15 min | Lab |
| Section 4: Evaluation & Decision Framework | 15 min | Demo + Theory |
| Wrap-up & Homework | 5 min | Markdown |

## What We'll Build Today

![Week 14 Overview Pipeline](charts/overview_pipeline.png)

**The punchline**: Fine-tuning a small model on synthetic data gives you
~90%+ accuracy at a fraction of the cost — FREE at inference, fast, and private.

## GPU Setup

**Runtime > Change runtime type > T4 GPU** (recommended for faster training)

# Section 0: Environment Setup

We'll use HuggingFace Transformers for model fine-tuning and the datasets
library for efficient data handling. GPU is recommended but not strictly required.

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES
# =============================================================================
# transformers: HuggingFace models and Trainer
# datasets: Efficient dataset handling
# accelerate: Optimized training on GPU
# scikit-learn: Evaluation metrics (precision, recall, F1)
# evaluate: HuggingFace evaluate library

!pip install -q transformers datasets accelerate scikit-learn evaluate

# =============================================================================
# IMPORTS
# =============================================================================
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
import evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

# =============================================================================
# VERIFY INSTALLATIONS
# =============================================================================
print("Library versions:")
print(f"  PyTorch:       {torch.__version__}")
print(f"  GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU device:    {torch.cuda.get_device_name(0)}")
    print(f"  GPU memory:    {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print("\n✅ All libraries installed successfully!")

In [ ]:
# =============================================================================
# TEST DATA — 50 Transactions from Week 13 (NEVER used for training)
# =============================================================================
# These are our held-out test set. We'll use them to evaluate the fine-tuned
# model and compare against all previous approaches.

test_transactions = [
    {"id": "TXN-001", "description": "Customer reports unauthorized wire transfer of $4,500 to unknown overseas account. No prior international transaction history. Transfer initiated at 3:47 AM local time.", "actual_label": "fraud"},
    {"id": "TXN-002", "description": "Regular monthly payment of $89.99 to Netflix streaming service. Consistent with 18-month subscription history. Payment from primary checking account.", "actual_label": "legitimate"},
    {"id": "TXN-003", "description": "Three consecutive ATM withdrawals totaling $1,500 in different cities within 2 hours. Card was reported lost the following day. Withdrawals at non-bank ATMs.", "actual_label": "fraud"},
    {"id": "TXN-004", "description": "Online purchase of $234.56 at Amazon.com for household electronics. Shipping to address on file. Customer has frequent Amazon purchase history.", "actual_label": "legitimate"},
    {"id": "TXN-005", "description": "Customer disputes charge of $2,100 at luxury jewelry store in Miami. Customer's location confirmed as Chicago at time of purchase. No travel alerts set.", "actual_label": "fraud"},
    {"id": "TXN-006", "description": "Automatic payroll direct deposit of $3,245.67 from employer ABC Corp. Matches bi-weekly pay schedule. Amount consistent with employment records.", "actual_label": "legitimate"},
    {"id": "TXN-007", "description": "Multiple small online purchases ($5-$15) at various digital stores within 30 minutes. None of these merchants appear in customer's history. Different IP addresses used.", "actual_label": "fraud"},
    {"id": "TXN-008", "description": "Grocery purchase of $67.23 at Whole Foods Market. Customer shops here weekly based on 2-year transaction history. Paid with debit card at POS terminal.", "actual_label": "legitimate"},
    {"id": "TXN-009", "description": "Account password changed and $8,200 transferred to a new payee within 15 minutes. Login originated from an IP address in a different country than the account holder's residence.", "actual_label": "fraud"},
    {"id": "TXN-010", "description": "Credit card used for $3,400 purchase at electronics store in Lagos, Nigeria. Cardholder has never traveled outside the United States. Card was not reported stolen.", "actual_label": "fraud"},
    {"id": "TXN-011", "description": "Five gift card purchases of $500 each at different Walmart locations within one hour. Customer has no prior gift card purchase history. All purchases made with the same debit card.", "actual_label": "fraud"},
    {"id": "TXN-012", "description": "Online gambling deposit of $2,000 to an unlicensed offshore betting site. Customer's account shows no prior gambling-related transactions. Deposit made at 4:12 AM.", "actual_label": "fraud"},
    {"id": "TXN-013", "description": "Wire transfer of $15,000 to a recently created account at a foreign bank. Transfer requested via phone call, but customer's voice did not match voiceprint on file.", "actual_label": "fraud"},
    {"id": "TXN-014", "description": "Rapid succession of 12 declined transactions followed by one approved transaction of $1,899 at an electronics retailer. Different card numbers attempted from same IP.", "actual_label": "fraud"},
    {"id": "TXN-015", "description": "Customer's debit card used for contactless payment of $750 at a gas station in Texas while customer was checked into a hotel in New York. No travel notification on file.", "actual_label": "fraud"},
    {"id": "TXN-016", "description": "Account drained of $6,300 through a series of Zelle transfers to three unknown recipients within 20 minutes. Customer claims they did not authorize the transfers.", "actual_label": "fraud"},
    {"id": "TXN-017", "description": "Purchase of $4,200 in cryptocurrency from an unregulated exchange using a newly added payment method. Account security questions were changed 30 minutes prior.", "actual_label": "fraud"},
    {"id": "TXN-018", "description": "Two simultaneous transactions: $1,100 at a restaurant in London and $890 at a store in Sydney. Physical card required at both locations. Cardholder resides in Boston.", "actual_label": "fraud"},
    {"id": "TXN-019", "description": "Refund of $2,500 processed to a different card than the original purchase. Original purchase was made 8 months ago. Refund requested through customer service chat.", "actual_label": "fraud"},
    {"id": "TXN-020", "description": "Cash advance of $3,000 at an ATM in a high-risk neighborhood at 2:30 AM. Customer's account has never had a cash advance. PIN was entered correctly on first attempt.", "actual_label": "fraud"},
    {"id": "TXN-021", "description": "Online purchase of $5,600 worth of designer handbags from a suspicious website with no SSL certificate. Shipping address differs from billing address and is a P.O. box.", "actual_label": "fraud"},
    {"id": "TXN-022", "description": "Authorized user added to account and immediately made a $7,500 purchase. The authorized user's identity could not be verified through standard KYC checks.", "actual_label": "fraud"},
    {"id": "TXN-023", "description": "Series of micro-transactions ($0.01 to $1.00) at 15 different online merchants within 5 minutes. Pattern consistent with card testing before a larger fraudulent purchase.", "actual_label": "fraud"},
    {"id": "TXN-024", "description": "Balance transfer of $12,000 to a new credit card opened the same day. Application used a slightly different spelling of the customer's name and a different phone number.", "actual_label": "fraud"},
    {"id": "TXN-025", "description": "Purchase of $950 airline ticket to a one-way international destination booked 2 hours before departure. Customer has no passport on file and no prior international travel.", "actual_label": "fraud"},
    {"id": "TXN-026", "description": "Monthly mortgage payment of $1,847.33 to Wells Fargo Home Mortgage. Amount unchanged for 3 years. Auto-debit from primary checking account on the 1st of each month.", "actual_label": "legitimate"},
    {"id": "TXN-027", "description": "Quarterly insurance premium of $412.00 to State Farm. Consistent with 5-year policy history. Payment matches scheduled auto-pay date.", "actual_label": "legitimate"},
    {"id": "TXN-028", "description": "Gas station purchase of $52.18 at Shell on Highway 101. Customer fills up at this location every Friday afternoon. Debit card used at pump with PIN.", "actual_label": "legitimate"},
    {"id": "TXN-029", "description": "Online subscription renewal of $14.99 for Spotify Premium. Same charge every month for the past 2 years. Billed to Visa ending in 4532.", "actual_label": "legitimate"},
    {"id": "TXN-030", "description": "Restaurant charge of $78.45 at Olive Garden in customer's hometown. Tip of 20% added. Customer dines here approximately twice per month.", "actual_label": "legitimate"},
    {"id": "TXN-031", "description": "Utility bill payment of $156.78 to ConEdison via online banking. Amount within normal seasonal range. Payment made 3 days before due date as usual.", "actual_label": "legitimate"},
    {"id": "TXN-032", "description": "Daycare tuition payment of $1,200.00 to Little Stars Learning Center. Same amount every two weeks since January. Matches enrollment records.", "actual_label": "legitimate"},
    {"id": "TXN-033", "description": "Pharmacy purchase of $23.45 at CVS near customer's home address. Customer has weekly prescription pickups at this location. Paid with FSA debit card.", "actual_label": "legitimate"},
    {"id": "TXN-034", "description": "Annual gym membership renewal of $599.00 at Planet Fitness. Same charge last year at the same time. Customer checks in 4-5 times per week.", "actual_label": "legitimate"},
    {"id": "TXN-035", "description": "Transfer of $500.00 to savings account at same bank. Customer makes this transfer on the 15th of every month. Part of automatic savings plan.", "actual_label": "legitimate"},
    {"id": "TXN-036", "description": "Online order of $145.67 at Target.com. Shipping to home address on file. Customer has Target Circle membership and shops online monthly.", "actual_label": "legitimate"},
    {"id": "TXN-037", "description": "Car payment of $423.56 to Toyota Financial Services. Amount matches lease agreement. Auto-pay set up 18 months ago, never missed a payment.", "actual_label": "legitimate"},
    {"id": "TXN-038", "description": "Coffee shop purchase of $6.75 at Starbucks near customer's office. Customer visits this location every weekday morning. Mobile order via app.", "actual_label": "legitimate"},
    {"id": "TXN-039", "description": "Charitable donation of $100.00 to American Red Cross. Customer makes this donation annually in December. Tax-deductible receipt issued.", "actual_label": "legitimate"},
    {"id": "TXN-040", "description": "Veterinary bill of $287.50 at Banfield Pet Hospital. Customer has a pet wellness plan. Visit scheduled 2 weeks ago for annual checkup.", "actual_label": "legitimate"},
    {"id": "TXN-041", "description": "Home improvement purchase of $342.89 at Home Depot. Customer recently purchased a home and has made 6 Home Depot purchases in the past month. In-store chip payment.", "actual_label": "legitimate"},
    {"id": "TXN-042", "description": "Monthly student loan payment of $567.89 to Navient. Same amount for 4 years. Auto-debit on the 5th of each month from checking account.", "actual_label": "legitimate"},
    {"id": "TXN-043", "description": "Dry cleaning pickup charge of $34.50 at Express Cleaners. Customer drops off clothes every Monday and picks up Wednesday. Store is 2 blocks from home.", "actual_label": "legitimate"},
    {"id": "TXN-044", "description": "Online purchase of $89.99 for annual antivirus software renewal from Norton. Same charge last year. License key sent to email on file.", "actual_label": "legitimate"},
    {"id": "TXN-045", "description": "Ride-share charge of $24.67 from Uber. Trip from customer's office to home address. Customer uses Uber 2-3 times per week for commute.", "actual_label": "legitimate"},
    {"id": "TXN-046", "description": "Wire transfer of $2,000 to customer's own account at another bank. Both accounts have been linked for 3 years. Transfer initiated via verified mobile app.", "actual_label": "legitimate"},
    {"id": "TXN-047", "description": "Hotel charge of $189.00 at Marriott in San Francisco. Customer has a confirmed reservation matching the dates. Corporate travel card used, expense report filed.", "actual_label": "legitimate"},
    {"id": "TXN-048", "description": "Lawn care service payment of $75.00 to GreenScape LLC. Same vendor, same amount, every two weeks from April through October for 3 years.", "actual_label": "legitimate"},
    {"id": "TXN-049", "description": "Birthday gift purchase of $65.00 at Barnes & Noble online. Shipping to a different address (gift recipient). Customer makes similar purchases around family birthdays.", "actual_label": "legitimate"},
    {"id": "TXN-050", "description": "Tollway auto-replenishment charge of $40.00 to I-PASS. Triggered when balance fell below $10 threshold. Customer commutes daily on the tollway.", "actual_label": "legitimate"},
]

test_df = pd.DataFrame(test_transactions)
print(f"Test set: {len(test_df)} transactions ({(test_df['actual_label']=='fraud').sum()} fraud, {(test_df['actual_label']=='legitimate').sum()} legitimate)")
print(f"⚠️ These are NEVER used for training — test set only!")

In [ ]:
# =============================================================================
# TRAINING DATA — From Week 13 Synthetic Data (or Fallback)
# =============================================================================

# Try to load Week 13's synthetic data
TRAIN_DATA_PATH = 'synthetic_fraud_data.csv'

if os.path.exists(TRAIN_DATA_PATH):
    train_df = pd.read_csv(TRAIN_DATA_PATH)
    print(f"✅ Loaded training data from Week 13: {len(train_df)} examples")
else:
    print("⚠️ Week 13 synthetic data not found. Generating fallback training data...")
    # Fallback: simple template-based training data (no API needed)
    fraud_templates = [
        "Unauthorized transaction of ${amount} at {merchant} in {city}. Customer was not in {city} at the time. No prior purchases at this merchant.",
        "Account compromised: ${amount} transferred to unknown account. Login from suspicious IP at {time} AM. Password changed without customer knowledge.",
        "Multiple rapid purchases totaling ${amount} across {count} merchants in {minutes} minutes. None of these merchants in customer history.",
        "Card used for ${amount} purchase in {country} while cardholder is in the US. No travel alert set. Transaction flagged by geographic monitoring.",
        "ATM withdrawal of ${amount} at {time} AM from unfamiliar location. Customer reports card was in their possession. PIN used correctly.",
    ]
    legit_templates = [
        "Regular monthly payment of ${amount} to {merchant}. Consistent with {months}-month subscription history. Auto-pay from checking account.",
        "Grocery purchase of ${amount} at {merchant} near customer's home. Weekly shopping pattern for {years} years. Debit card with PIN.",
        "Online purchase of ${amount} at {merchant}. Shipping to address on file. Customer is a frequent shopper at this retailer.",
        "Bill payment of ${amount} to {merchant} via online banking. Amount within normal range. Paid {days} days before due date as usual.",
        "Direct deposit of ${amount} from employer {merchant}. Matches bi-weekly pay schedule. Same amount for {months} months.",
    ]

    import random
    random.seed(42)
    merchants_fraud = ['ElectroMax', 'LuxuryGems', 'CryptoExchange', 'QuickCash ATM', 'TechWorld']
    merchants_legit = ['Netflix', 'Whole Foods', 'Amazon', 'ConEdison', 'ABC Corp']
    cities = ['Lagos', 'Miami', 'London', 'Bucharest', 'São Paulo']
    countries = ['Nigeria', 'Romania', 'Brazil', 'Philippines', 'Russia']

    train_data = []
    for i in range(50):
        tmpl = random.choice(fraud_templates)
        desc = tmpl.format(
            amount=random.randint(500, 15000),
            merchant=random.choice(merchants_fraud),
            city=random.choice(cities),
            time=random.randint(1, 5),
            count=random.randint(3, 12),
            minutes=random.randint(5, 30),
            country=random.choice(countries),
        )
        train_data.append({'description': desc, 'label': 'fraud'})

    for i in range(50):
        tmpl = random.choice(legit_templates)
        desc = tmpl.format(
            amount=f"{random.randint(10, 3000)}.{random.randint(0,99):02d}",
            merchant=random.choice(merchants_legit),
            months=random.randint(6, 36),
            years=random.randint(1, 5),
            days=random.randint(1, 5),
        )
        train_data.append({'description': desc, 'label': 'legitimate'})

    train_df = pd.DataFrame(train_data)
    print(f"✅ Generated fallback training data: {len(train_df)} examples")

# Show summary
print(f"\nTraining data: {len(train_df)} examples")
print(f"  Fraud:      {(train_df['label'] == 'fraud').sum()}")
print(f"  Legitimate: {(train_df['label'] == 'legitimate').sum()}")
print(f"\nSample fraud:")
print(f"  {train_df[train_df['label']=='fraud'].iloc[0]['description'][:100]}...")
print(f"\nSample legitimate:")
print(f"  {train_df[train_df['label']=='legitimate'].iloc[0]['description'][:100]}...")

# Section 1: Transfer Learning Concepts

## The Fine-Tuning Spectrum

![Fine-Tuning Spectrum](charts/finetuning_spectrum.png)

**Less Data/Compute ◄──────────────────────────► More Data/Compute**

## What Is Transfer Learning?

A pre-trained model like DistilBERT has already learned **how language works**
from reading billions of words. It understands grammar, word relationships,
and context. We don't need to teach it English — we just need to teach it
**our specific task**: classifying transactions as fraud or legitimate.

**Analogy**: Hiring someone who already speaks fluent English and teaching
them banking terminology. Much faster than teaching a baby from scratch.

## Why DistilBERT?

| Property | Value |
|----------|-------|
| Parameters | 67 million |
| Size vs BERT | 40% smaller |
| Speed vs BERT | 60% faster |
| Performance vs BERT | Retains 97% of BERT's quality |
| Max input length | 512 tokens |

In Week 12, we used `distilbert-base-uncased-finetuned-sst-2-english` as a
**sentiment proxy** for fraud detection. It worked OK (~62% accuracy) but
wasn't trained for our task. Now we'll train our OWN DistilBERT for fraud.

In [ ]:
# =============================================================================
# DEMO: Load DistilBERT for Sequence Classification
# =============================================================================

MODEL_NAME = 'distilbert-base-uncased'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model with a classification head (2 labels: fraud, legitimate)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'legitimate', 1: 'fraud'},
    label2id={'legitimate': 0, 'fraud': 1}
)

# Examine the architecture
print("Model Architecture:")
print(f"  Base model: {MODEL_NAME}")
print(f"  Classification labels: legitimate (0), fraud (1)")
print()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
classifier_params = sum(p.numel() for n, p in model.named_parameters() if 'classifier' in n or 'pre_classifier' in n)

print(f"Parameter counts:")
print(f"  Total:      {total_params:>12,}")
print(f"  Trainable:  {trainable_params:>12,} (all — we'll train everything)")
print(f"  Classifier: {classifier_params:>12,} (the new head we added)")
print(f"  Encoder:    {total_params - classifier_params:>12,} (pre-trained language knowledge)")
print(f"\n💡 The classifier head is tiny ({classifier_params:,} params) compared to the")
print(f"   encoder ({total_params - classifier_params:,} params). But even fine-tuning the")
print(f"   encoder is fast on a T4 GPU — only 67M params total.")

> **Think About It**: In Week 12, we used DistilBERT-SST2 (trained on movie
> sentiment) as a PROXY for fraud detection — mapping NEGATIVE to fraud,
> POSITIVE to legitimate. It got ~62% accuracy. Now we'll train our own DistilBERT
> specifically for fraud. Why should this work better? What's the difference
> between "this text sounds negative" and "this transaction is fraudulent"?

# Section 2: Tokenization & Data Preparation

Before fine-tuning, we need to convert our text descriptions into the format
DistilBERT expects: token IDs, attention masks, and integer labels.

In [ ]:
# =============================================================================
# DEMO: Tokenize a Single Transaction
# =============================================================================

sample_text = train_df.iloc[0]['description']
print(f"Original text ({len(sample_text.split())} words):")
print(f'  "{sample_text[:100]}..."')

# Tokenize
encoded = tokenizer(
    sample_text,
    padding='max_length',
    truncation=True,
    max_length=128,      # 128 tokens is enough for transaction descriptions
    return_tensors='pt'
)

print(f"\nTokenized:")
print(f"  input_ids shape:      {encoded['input_ids'].shape}")
print(f"  attention_mask shape: {encoded['attention_mask'].shape}")
print(f"  Number of real tokens: {encoded['attention_mask'].sum().item()}")
print(f"  Padding tokens:        {128 - encoded['attention_mask'].sum().item()}")

# Show first 20 tokens
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0][:20])
print(f"\nFirst 20 tokens: {tokens}")
print(f"\n💡 DistilBERT uses WordPiece tokenization — words get split into subwords.")
print(f"   [CLS] marks the start, [SEP] marks the end, [PAD] fills to max_length.")

In [ ]:
# =============================================================================
# DEMO: Create HuggingFace Datasets for Training
# =============================================================================

# Label mapping
LABEL2ID = {'legitimate': 0, 'fraud': 1}
ID2LABEL = {0: 'legitimate', 1: 'fraud'}

# Prepare training data
train_df['label_id'] = train_df['label'].map(LABEL2ID)

# Split training data: 80% train, 20% validation
train_split, val_split = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df['label']
)

print(f"Train split: {len(train_split)} examples ({(train_split['label']=='fraud').sum()} fraud)")
print(f"Val split:   {len(val_split)} examples ({(val_split['label']=='fraud').sum()} fraud)")
print(f"Test set:    {len(test_df)} examples (held out from Week 13)")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_dict({
    'text': train_split['description'].tolist(),
    'label': train_split['label_id'].tolist()
})
val_dataset = Dataset.from_dict({
    'text': val_split['description'].tolist(),
    'label': val_split['label_id'].tolist()
})

# Prepare test set too
test_df['label_id'] = test_df['actual_label'].map(LABEL2ID)
test_dataset = Dataset.from_dict({
    'text': test_df['description'].tolist(),
    'label': test_df['label_id'].tolist()
})

print(f"\nHuggingFace Datasets created:")
print(f"  train: {train_dataset}")
print(f"  val:   {val_dataset}")
print(f"  test:  {test_dataset}")

In [ ]:
# =============================================================================
# DEMO: Tokenize Full Dataset with .map()
# =============================================================================

def tokenize_function(examples):
    """Tokenize a batch of examples."""
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )


# Tokenize all splits
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenized datasets ready:")
print(f"  train: {len(train_tokenized)} examples")
print(f"  val:   {len(val_tokenized)} examples")
print(f"  test:  {len(test_tokenized)} examples")
print(f"\nColumns: {train_tokenized.column_names}")

In [ ]:
# =============================================================================
# DEMO: Inspect Tokenization Quality and Padding Behavior
# =============================================================================
# Before you build your own pipeline in Lab 1, let's look at what happens
# when we batch tokenized examples together with dynamic padding.

# Show token length distribution
token_lengths = [len(train_tokenized[i]['input_ids']) for i in range(len(train_tokenized))]
print("Token Length Statistics:")
print(f"  Min:    {min(token_lengths)}")
print(f"  Max:    {max(token_lengths)}")
print(f"  Mean:   {np.mean(token_lengths):.1f}")
print(f"  Median: {np.median(token_lengths):.1f}")

# Visualize distribution
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(token_lengths, bins=20, color='#3498db', edgecolor='black', alpha=0.7)
ax.axvline(128, color='red', linestyle='--', label='max_length=128')
ax.set_xlabel('Number of Tokens')
ax.set_ylabel('Count')
ax.set_title('Token Length Distribution (Training Set)')
ax.legend()
plt.tight_layout()
plt.show()

# Show how DataCollator pads a mini-batch
sample_batch = [train_tokenized[i] for i in range(4)]
collated = data_collator(sample_batch)
print(f"\nDynamic padding in action (batch of 4):")
for i in range(4):
    real_tokens = collated['attention_mask'][i].sum().item()
    total_tokens = len(collated['input_ids'][i])
    padding = total_tokens - real_tokens
    print(f"  Example {i}: {real_tokens} real tokens + {padding} padding = {total_tokens} total")

## Lab 1: Prepare Your Dataset (15 minutes)

### Your Task

Create a complete data preparation pipeline: load data, split, tokenize,
and verify everything is ready for training.

### Steps

1. **Verify label distribution**: Check that train/val splits are balanced
2. **Experiment with max_length**: Try 64 vs 128 vs 256 — which truncates the least?
3. **Create a data collator** for dynamic padding (more efficient than max_length padding)
4. **Verify a sample batch**: Decode tokens back to text to confirm correctness

### Expected Output

- Print label distribution per split
- Comparison of truncation rates at different max_lengths
- DataCollatorWithPadding object created
- Decoded sample matching original text

### Homework Extension

After class: analyze which transactions get truncated at max_length=128.
Are fraud descriptions longer than legitimate ones? Does truncation affect
classification accuracy?

In [ ]:
# =============================================================================
# SOLUTION: LAB 1 — PREPARE YOUR DATASET
# =============================================================================

# Print label distribution for train and val splits
train_labels = pd.Series([ID2LABEL[l] for l in train_tokenized['label'].numpy()])
val_labels = pd.Series([ID2LABEL[l] for l in val_tokenized['label'].numpy()])

print("Label Distribution:")
print(f"  Train: {dict(train_labels.value_counts())}")
print(f"  Val:   {dict(val_labels.value_counts())}")

# Compare truncation at different max_lengths
print("\nTruncation Analysis:")
for max_len in [64, 128, 256]:
    truncated = 0
    for text in train_df['description']:
        tokens = tokenizer(text, truncation=False)
        if len(tokens['input_ids']) > max_len:
            truncated += 1
    pct = truncated / len(train_df) * 100
    print(f"  max_length={max_len}: {truncated}/{len(train_df)} truncated ({pct:.1f}%)")

# Create DataCollatorWithPadding (more efficient than fixed padding)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Decode a sample back to text
sample_ids = train_tokenized[0]['input_ids']
sample_decoded = tokenizer.decode(sample_ids, skip_special_tokens=True)
print(f"\nSample decoded: \"{sample_decoded[:100]}...\"")
print(f"Original:       \"{train_split.iloc[0]['description'][:100]}...\"")

print("\n✅ Data preparation complete!")
print(f"   Data collator: {type(data_collator).__name__}")

# Section 3: Fine-Tuning DistilBERT

Now the main event: we'll fine-tune DistilBERT to classify fraud.

The HuggingFace `Trainer` handles the training loop for us:
- Forward pass (model predictions)
- Loss calculation (cross-entropy)
- Backward pass (gradient computation)
- Parameter update (optimizer step)
- Evaluation on validation set
- Checkpointing (save best model)

We just need to configure it.

In [ ]:
# =============================================================================
# DEMO: Define Training Configuration
# =============================================================================

training_args = TrainingArguments(
    output_dir='./fraud-classifier',           # Save checkpoints here
    num_train_epochs=3,                        # 3 passes through training data
    per_device_train_batch_size=16,            # Batch size per GPU
    per_device_eval_batch_size=16,             # Batch size for evaluation
    eval_strategy='epoch',                     # Evaluate after each epoch
    save_strategy='epoch',                     # Save checkpoint after each epoch
    learning_rate=2e-5,                        # Standard for BERT fine-tuning
    weight_decay=0.01,                         # Regularization
    load_best_model_at_end=True,               # Load best model when done
    metric_for_best_model='accuracy',          # Use accuracy to pick best
    logging_steps=10,                          # Log every 10 steps
    report_to='none',                          # Don't report to W&B etc.
    fp16=torch.cuda.is_available(),            # Use mixed precision on GPU
)

print("Training Configuration:")
print(f"  Epochs:        {training_args.num_train_epochs}")
print(f"  Batch size:    {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16:          {training_args.fp16}")
print(f"  Device:        {'GPU' if torch.cuda.is_available() else 'CPU'}")

# Estimate training time
steps_per_epoch = len(train_tokenized) // training_args.per_device_train_batch_size
total_steps = steps_per_epoch * training_args.num_train_epochs
print(f"\n  Steps/epoch:   {steps_per_epoch}")
print(f"  Total steps:   {total_steps}")
print(f"  Est. time:     ~{total_steps * 0.3:.0f}s on T4 GPU")

In [ ]:
# =============================================================================
# DEMO: Define Evaluation Metrics
# =============================================================================

accuracy_metric = evaluate.load('accuracy')


def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, F1 during training."""
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


# =============================================================================
# DEMO: Create Trainer and TRAIN!
# =============================================================================

# Re-load model fresh (in case we're re-running)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label=ID2LABEL, label2id=LABEL2ID
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting training...")
print("=" * 50)
train_result = trainer.train()
print("=" * 50)
print(f"\n✅ Training complete!")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"  Training time: {train_result.metrics['train_runtime']:.0f}s")

In [ ]:
# =============================================================================
# DEMO: Plot Training Metrics
# =============================================================================

# Extract training history
history = trainer.state.log_history

# Separate training and eval logs
train_logs = [h for h in history if 'loss' in h and 'eval_loss' not in h]
eval_logs = [h for h in history if 'eval_loss' in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Training loss
if train_logs:
    steps = [h['step'] for h in train_logs]
    losses = [h['loss'] for h in train_logs]
    ax1.plot(steps, losses, 'b-', linewidth=2)
    ax1.set_xlabel('Training Step')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')

# Validation metrics per epoch
if eval_logs:
    epochs = list(range(1, len(eval_logs) + 1))
    val_acc = [h['eval_accuracy'] for h in eval_logs]
    val_f1 = [h['eval_f1'] for h in eval_logs]
    ax2.plot(epochs, val_acc, 'g-o', label='Accuracy', linewidth=2)
    ax2.plot(epochs, val_f1, 'r-o', label='F1 Score', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Score')
    ax2.set_title('Validation Metrics')
    ax2.legend()
    ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

# Print final metrics
if eval_logs:
    final = eval_logs[-1]
    print(f"Final validation metrics (epoch {len(eval_logs)}):")
    print(f"  Accuracy:  {final['eval_accuracy']:.1%}")
    print(f"  Precision: {final['eval_precision']:.1%}")
    print(f"  Recall:    {final['eval_recall']:.1%}")
    print(f"  F1 Score:  {final['eval_f1']:.1%}")

In [ ]:
# =============================================================================
# DEMO: Evaluate on the 50 Test Transactions
# =============================================================================
# These are the SAME 50 transactions from Week 13 — never seen during training.

test_results = trainer.predict(test_tokenized)
test_preds = np.argmax(test_results.predictions, axis=-1)
test_labels = test_results.label_ids

# Classification report
print("Test Set Results (50 transactions)")
print("=" * 50)
print(classification_report(
    test_labels, test_preds,
    target_names=['legitimate', 'fraud']
))

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['legitimate', 'fraud'])
disp.plot(ax=ax, cmap='Blues')
ax.set_title('Fine-Tuned DistilBERT — Confusion Matrix')
plt.tight_layout()
plt.show()

# Show specific errors
test_df['predicted'] = [ID2LABEL[p] for p in test_preds]
test_df['correct'] = test_df['predicted'] == test_df['actual_label']
errors = test_df[~test_df['correct']]
print(f"\nErrors ({len(errors)}/{len(test_df)}):")
for _, row in errors.iterrows():
    print(f"  {row['id']}: predicted '{row['predicted']}', actual '{row['actual_label']}'")
    print(f"    \"{row['description'][:80]}...\"")

> **Think About It**: We fine-tuned on synthetic data generated by an LLM
> (Claude Haiku in Week 13). This is a form of **model distillation** — a large
> model's knowledge transferred to a smaller one via its labels. What risks
> does this introduce? What if the synthetic data has biases that real fraud
> data doesn't? How would you validate on REAL transactions before deploying?

In [ ]:
# =============================================================================
# DEMO: Understanding Learning Rate Impact (Quick Experiment)
# =============================================================================
# Before Lab 2, let's see WHY learning rate matters so much for fine-tuning.
# We'll train for just a few steps with an absurdly high LR vs our standard LR.

# Dangerous LR: 1e-3 (too high for fine-tuning)
bad_args = TrainingArguments(
    output_dir='./fraud-bad-lr', num_train_epochs=1,
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    eval_strategy='epoch', learning_rate=1e-3,  # 50x too high!
    weight_decay=0.01, logging_steps=5, report_to='none',
    fp16=torch.cuda.is_available(),
)

bad_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
)
bad_trainer = Trainer(
    model=bad_model, args=bad_args,
    train_dataset=train_tokenized, eval_dataset=val_tokenized,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Training with lr=1e-3 (intentionally too high)...")
bad_trainer.train()
bad_eval = bad_trainer.evaluate()

print(f"\n{'='*50}")
print(f"lr=2e-5 (our demo):  accuracy = {eval_logs[-1]['eval_accuracy']:.1%}")
print(f"lr=1e-3 (too high):  accuracy = {bad_eval['eval_accuracy']:.1%}")
print(f"{'='*50}")
print("\nToo-high LR destroys the pre-trained weights — the model 'forgets'")
print("what it learned during pre-training. This is called 'catastrophic forgetting'.")
print("That's why fine-tuning uses MUCH smaller LRs (1e-5 to 5e-5) than training from scratch.")

## Lab 2: Fine-Tune Your Classifier (15 minutes)

### Your Task

Experiment with different training configurations to see how they affect
accuracy on the 50-transaction test set.

### Steps

1. **Experiment with learning rate**: Try 1e-5 vs 2e-5 vs 5e-5
2. **Train for each configuration** (1 epoch each for speed)
3. **Record test accuracy** for each configuration
4. **Create a summary table**: learning rate, train loss, test accuracy, F1
5. **Create a confusion matrix** for your best model

### Expected Output

- Table with 3 rows (one per learning rate)
- Confusion matrix for the best model
- Best learning rate identified

### Homework Extension

After class: try varying epochs (1 vs 3 vs 5 vs 10) with your best learning rate.
Plot accuracy vs epochs. At what point does the model overfit (val accuracy drops)?

In [ ]:
# =============================================================================
# SOLUTION: LAB 2 — FINE-TUNE YOUR CLASSIFIER
# =============================================================================

learning_rates = [1e-5, 2e-5, 5e-5]
lab2_results = []

for lr in learning_rates:
    print(f"\n{'='*40}")
    print(f"Training with lr={lr}")
    print(f"{'='*40}")

    # Create TrainingArguments with this learning rate (1 epoch for speed)
    args = TrainingArguments(
        output_dir=f'./fraud-classifier-lr{lr}',
        num_train_epochs=1,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy='epoch',
        learning_rate=lr,
        weight_decay=0.01,
        logging_steps=50,
        report_to='none',
        fp16=torch.cuda.is_available(),
    )

    # Re-load fresh model
    fresh_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2,
        id2label=ID2LABEL, label2id=LABEL2ID
    )

    # Create Trainer and train
    lab_trainer = Trainer(
        model=fresh_model,
        args=args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    lab_trainer.train()

    # Evaluate on test set
    test_res = lab_trainer.predict(test_tokenized)
    preds = np.argmax(test_res.predictions, axis=-1)
    test_acc = accuracy_score(test_res.label_ids, preds)
    _, _, test_f1, _ = precision_recall_fscore_support(
        test_res.label_ids, preds, average='binary'
    )
    train_loss = lab_trainer.state.log_history[-2].get('loss', 0)

    lab2_results.append({
        'learning_rate': lr,
        'train_loss': train_loss,
        'test_accuracy': test_acc,
        'test_f1': test_f1,
    })
    print(f"  Test accuracy: {test_acc:.1%}, F1: {test_f1:.3f}")

# Summary table
summary_df = pd.DataFrame(lab2_results)
print(f"\n{'='*60}")
print("Learning Rate Experiment Summary:")
display(summary_df)

# Best model confusion matrix
best_idx = summary_df['test_accuracy'].idxmax()
best_lr = summary_df.loc[best_idx, 'learning_rate']
print(f"\nBest: lr={best_lr}, accuracy={summary_df.loc[best_idx, 'test_accuracy']:.1%}")
print("\nLab 2 complete!")

# Section 4: The Grand Comparison — When to Fine-Tune?

Let's put it all together. We've tried multiple approaches across 4 weeks.
How do they compare?

In [ ]:
# =============================================================================
# DEMO: Grand Comparison — All Approaches Across 4 Weeks
# =============================================================================

# Results from previous weeks (approximate — students can fill in actuals)
comparison = pd.DataFrame([
    {'Week': 11, 'Approach': 'OpenAI GPT-4o-mini', 'Type': 'Cloud API (prompted)',
     'Accuracy': 0.94, 'Latency': '~1.0s', 'Cost_per_call': '$0.0003', 'Training': 'None'},
    {'Week': 12, 'Approach': 'GPT-2 (local)', 'Type': 'Local (prompted)',
     'Accuracy': 0.50, 'Latency': '~0.5s', 'Cost_per_call': 'Free', 'Training': 'None'},
    {'Week': 12, 'Approach': 'Flan-T5 prompted', 'Type': 'Local (prompted)',
     'Accuracy': 0.75, 'Latency': '~0.3s', 'Cost_per_call': 'Free', 'Training': 'None'},
    {'Week': 12, 'Approach': 'DistilBERT-SST2 proxy', 'Type': 'Local (fine-tuned for other task)',
     'Accuracy': 0.62, 'Latency': '~0.05s', 'Cost_per_call': 'Free', 'Training': 'None'},
    {'Week': 13, 'Approach': 'Bedrock Claude Haiku', 'Type': 'Managed AWS (prompted)',
     'Accuracy': 0.94, 'Latency': '~0.8s', 'Cost_per_call': '$0.0002', 'Training': 'None'},
    {'Week': 13, 'Approach': 'Bedrock Nova Lite', 'Type': 'Managed AWS (prompted)',
     'Accuracy': 0.88, 'Latency': '~0.5s', 'Cost_per_call': '$0.00003', 'Training': 'None'},
    {'Week': 14, 'Approach': 'DistilBERT fine-tuned', 'Type': 'Local (fine-tuned for fraud)',
     'Accuracy': test_df['correct'].mean(), 'Latency': '~0.05s', 'Cost_per_call': 'Free*',
     'Training': f'{train_result.metrics["train_runtime"]:.0f}s on T4'},
])

print("Grand Comparison: Fraud Classification Across 4 Weeks")
print("=" * 80)
display(comparison)

print("\n* Free at inference — training cost was GPU compute time only")
print("\nKey insight: Fine-tuning a small model (67M params) gives you")
print("   near-cloud-API accuracy at local-model speed and cost.")

In [ ]:
# =============================================================================
# DEMO: When to Fine-Tune vs When to Prompt
# =============================================================================

decision_framework = """
DECISION FRAMEWORK: Fine-Tune or Prompt?
=========================================

Choose PROMPT ENGINEERING when:
  You have < 50 labeled examples
  The task changes frequently
  You need reasoning/explanations (not just labels)
  Budget allows cloud API costs at your volume
  You need it working TODAY (no training time)

Choose FINE-TUNING when:
  You have 100+ labeled examples (or can generate synthetic data)
  The task is stable (won't change monthly)
  You need fast inference (real-time scoring)
  You need to minimize per-call costs at scale
  You need a consistent output format (just labels, no variance)
  Data privacy requires keeping everything on-premise

The HYBRID approach (what we did!):
  1. Use a big model (Bedrock Claude) to GENERATE training data
  2. Fine-tune a small model (DistilBERT) on that data
  3. Deploy the small model for production inference
  -> Best of both worlds: big model quality, small model cost
"""

print(decision_framework)

> **Think About It**: We used Claude Haiku (via Bedrock) to generate synthetic
> training data, then fine-tuned DistilBERT on that data. This is a form of
> **knowledge distillation**. The fine-tuned model can now run without any API
> calls. At 1 million transactions per month, what's the cost difference between
> calling Bedrock Claude Haiku vs running your fine-tuned DistilBERT locally?

## A Note on PEFT, LoRA, and Distillation

In this session, we fine-tuned ALL of DistilBERT's parameters (67M). This works
great for small models on a T4 GPU. But there are two important advanced techniques
to be aware of:

### Parameter-Efficient Fine-Tuning (PEFT/LoRA)

**LoRA** lets you fine-tune only ~1% of a model's parameters by adding small
"adapter" layers. This makes it possible to fine-tune models with billions of
parameters on consumer GPUs.

| Technique | What It Does | When to Use |
|-----------|-------------|-------------|
| **Full fine-tuning** (today) | Updates all parameters | Small models (< 1B params) |
| **LoRA** | Adds small adapter matrices | Medium models (1B-7B params) |
| **QLoRA** | LoRA + 4-bit quantization | Large models (7B+ params), limited GPU |

See: `week_14_optional_peft_lora.ipynb`

### Knowledge Distillation

What we did today (using a large model's outputs to train a smaller one) is
actually a form of **distillation**. Formal distillation goes further — training
the student model to match the teacher's probability distributions, not just
its hard labels.

See: `week_14_optional_distillation.ipynb`

In [ ]:
# =============================================================================
# QUICK PEEK: What LoRA Code Looks Like (don't worry about details!)
# =============================================================================
# This is just a preview — the optional notebook goes deeper.
# Notice how small the adapter is compared to the full model.

print("Full fine-tuning (what we did today):")
print(f"  Trainable parameters: {trainable_params:,} (100% of model)")
print()
print("LoRA fine-tuning (optional notebook):")
lora_params = 2 * 768 * 8 * 6  # rank=8, 6 attention layers (approximate)
print(f"  Trainable parameters: ~{lora_params:,} ({lora_params/total_params:.1%} of model)")
print(f"  Frozen parameters:    ~{total_params - lora_params:,}")
print()
print("Key insight: LoRA trains <1% of parameters but achieves similar accuracy.")
print("   Essential for fine-tuning models with billions of parameters.")

# Summary: What We Learned Today

## Key Takeaways

### Transfer Learning
- Pre-trained models already understand language
- We just add a task-specific "head" and fine-tune on our data
- DistilBERT: 67M params, trains in minutes on a T4 GPU

### The Full Pipeline
1. **Generate synthetic data** with a large model (Week 13, Bedrock)
2. **Tokenize and prepare** with HuggingFace (today)
3. **Fine-tune** with HuggingFace Trainer (today)
4. **Evaluate** against prompted approaches (today)

### When to Fine-Tune
- Lots of labeled data + stable task + need speed/low cost → fine-tune
- Few examples + changing task + need explanations → prompt engineering
- Hybrid: big model generates labels → fine-tune small model

## The Bigger Picture

![Week Progression](charts/week_progression.png)

| Week | What We Did |
|------|------------|
| 11 | Cloud APIs (OpenAI, Anthropic), prompting patterns |
| 12 | Local HuggingFace models, evaluation, error analysis |
| 13 | Bedrock managed models, Knowledge Bases, synthetic data, DeepEval |
| **14** | **Fine-tuned DistilBERT fraud classifier (transfer learning)** |
| 15-16 | Agentic AI — ReAct agents, LangChain, multi-agent systems |

# Homework & Optional Labs

## Homework (Complete before next session)

### Homework 1: Hyperparameter Sweep
Try 5 combinations of learning rate x epochs:
- lr in {1e-5, 2e-5, 5e-5}, epochs in {1, 3, 5}
Create a heatmap showing accuracy for each combination.

### Homework 2: More Training Data
If you have Week 13's Bedrock access, generate 200 more synthetic transactions.
Retrain with the larger dataset. Does accuracy improve? By how much?

### Homework 3: Error Analysis
Take the test set errors from your best model. Why did they fail?
Are the misclassified transactions ambiguous even for humans?
Compare error patterns to Week 12's prompt-based error analysis.

## Optional Labs

### Optional 1: PEFT/LoRA Fine-Tuning
See `week_14_optional_peft_lora.ipynb` for:
- Install peft and bitsandbytes
- LoRA configuration (r, alpha, target_modules)
- Fine-tune Flan-T5 with LoRA adapters
- Compare: full fine-tuning vs LoRA (param count, memory, accuracy)

### Optional 2: Knowledge Distillation
See `week_14_optional_distillation.ipynb` for:
- Formal distillation with soft labels (temperature scaling)
- Train a student model on teacher probability distributions
- Compare: hard labels vs soft labels training
- When distillation outperforms standard fine-tuning

# Great Work Today!

You've completed Week 14 of the AI for Data Scientists Academy.

**What you accomplished:**
- Prepared text data for fine-tuning with HuggingFace tokenizers
- Fine-tuned DistilBERT for binary fraud classification
- Compared fine-tuned model against prompted approaches from Weeks 11-13
- Built a decision framework for when to fine-tune vs prompt

## Coming Up Next

- **Weeks 15-16**: Agentic AI — ReAct agents, LangChain, LangGraph
- **Weeks 17-18**: RAG — build retrieval pipelines (beyond Bedrock Knowledge Bases)
- **Weeks 19-20**: MLOps — DVC, CI/CD, monitoring

## Resources

- [HuggingFace Fine-Tuning Tutorial](https://huggingface.co/docs/transformers/training)
- [HuggingFace Trainer Documentation](https://huggingface.co/docs/transformers/main_classes/trainer)
- [DistilBERT Paper](https://arxiv.org/abs/1910.01108)
- [PEFT Library](https://huggingface.co/docs/peft)

See you in Week 15!